In [ ]:
# mini_gpt_bpe.py
# Minimal GPT with Byte Pair Encoding (BPE) tokenization

import math, torch, requests
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load larger corpus (Shakespeare tiny dataset) ---
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
corpus = requests.get(url).text

# --- Train BPE tokenizer ---
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
trainer = BpeTrainer(
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"], vocab_size=5000
)
tokenizer.pre_tokenizer = Whitespace()
tokenizer.train_from_iterator(corpus.splitlines(), trainer=trainer)


# encode/decode helpers
def encode(text):
    return tokenizer.encode(text).ids


def decode(ids):
    return tokenizer.decode(ids)


# turn corpus into ids
data = torch.tensor(encode(corpus), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]


# --- Model config ---
@dataclass
class CFG:
    block: int = 64
    emb: int = 128
    heads: int = 4
    layers: int = 2
    dropout: float = 0.1


cfg = CFG()


def get_batch(split, bs=12):
    src = train_data if split == "train" else val_data
    ix = torch.randint(0, len(src) - cfg.block - 1, (bs,))
    x = torch.stack([src[i : i + cfg.block] for i in ix]).to(device)
    y = torch.stack([src[i + 1 : i + 1 + cfg.block] for i in ix]).to(device)
    return x, y


# --- Transformer pieces (same as before) ---
class LN(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.g = nn.Parameter(torch.ones(n))
        self.b = nn.Parameter(torch.zeros(n))

    def forward(self, x):
        m = x.mean(-1, keepdim=True)
        v = x.var(-1, unbiased=False, keepdim=True)
        return self.g * (x - m) / torch.sqrt(v + 1e-5) + self.b


class Attn(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(cfg.emb, 3 * cfg.emb)
        self.proj = nn.Linear(cfg.emb, cfg.emb)
        self.drop = nn.Dropout(cfg.dropout)
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(cfg.block, cfg.block)).view(
                1, 1, cfg.block, cfg.block
            ),
        )
        self.heads = cfg.heads
        self.hdim = cfg.emb // cfg.heads

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.heads, self.hdim).transpose(1, 2)
        k = k.view(B, T, self.heads, self.hdim).transpose(1, 2)
        v = v.view(B, T, self.heads, self.hdim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / (self.hdim**0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.proj(y))


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.emb, 4 * cfg.emb),
            nn.GELU(),
            nn.Linear(4 * cfg.emb, cfg.emb),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = LN(cfg.emb)
        self.ln2 = LN(cfg.emb)
        self.att = Attn()
        self.mlp = MLP()

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.tok = nn.Embedding(vocab, cfg.emb)
        self.pos = nn.Embedding(cfg.block, cfg.emb)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block() for _ in range(cfg.layers)])
        self.lnf = LN(cfg.emb)
        self.head = nn.Linear(cfg.emb, vocab, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        x = self.drop(x)
        for bl in self.blocks:
            x = bl(x)
        x = self.lnf(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -cfg.block :])
            logits = logits[:, -1, :]
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx


# --- Training and sampling ---
def main():
    vocab = tokenizer.get_vocab_size()
    model = MiniGPT(vocab).to(device)
    opt = torch.optim.AdamW(
        model.parameters(), lr=4e-4, betas=(0.9, 0.95), weight_decay=1e-2
    )

    max_iters = 5000
    model.train()
    for step in range(1, max_iters + 1):
        xb, yb = get_batch("train", bs=16)
        _, loss = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step % 50 == 0:
            print(f"step {step:3d} | loss {loss.item():.3f}")

    # Generate text
    start = "Not"
    start_ids = torch.tensor([encode(start)], dtype=torch.long, device=device)
    gen_ids = model.generate(start_ids, max_new_tokens=100)[0].tolist()
    print("\n=== SAMPLE ===\n")
    print(decode(gen_ids))


if __name__ == "__main__":
    main()




step  50 | loss 6.790
step 100 | loss 6.484
step 150 | loss 6.393
step 200 | loss 6.368
step 250 | loss 6.283
step 300 | loss 6.083
step 350 | loss 6.047
step 400 | loss 6.108
step 450 | loss 5.947
step 500 | loss 5.919

=== SAMPLE ===

Not not ness dishonour hand ' ll , CATESBY e , Sirrah us : Fie oracle . army : Come me that I prithee , you hath : he is the help , found That vel ' tis , to the way , sir ! Nay , dur yet holy it I must ? First well like to of it did forward pa ' s advance : He ? That you bloody than co point : Ay : ens athe ! it Must an us here : What have this sets o , To slave here bl sense . What I
